# Avaliação 2 — Prática individual · Encontro 12

**Disciplina:** Métodos e Técnicas de Pesquisa Quantitativa — Administração/UFMA

**Instruções — leia antes de começar:**
- Duração: **150 minutos**. Consulta ao material da disciplina **permitida**; comunicação
com colegas, **não**;
- Preencha seu nome e matrícula abaixo e execute as células de preparação;
- Esta avaliação cobre os encontros **9, 10 e 11** (descritivas, gráficos, inferência);
- As três tarefas valem **100 pontos** (20 + 25 + 30 + 25);
- Cada resposta final deve estar **escrita no notebook**, junto dos resultados — comentar
o que o resultado significa vale pontos;
- Ao final: salve, compartilhe o link com o professor e verifique se **todas as células
executadas** aparecem com resultado.

In [ ]:
# === PREENCHA seus dados ===
nome = ""
matricula = ""

print("Estudante:", nome, "| Matrícula:", matricula)

## Preparação — base CVM (execute, não altere)

As células abaixo constroem a base com a qual você trabalhará: receita, lucro líquido,
margem e log-receita das companhias abertas (DFP 2024). Se o download falhar, a célula de
contingência carrega o arquivo local.

In [ ]:
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from scipy import stats

url_dfp = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/dfp_cia_aberta_2024.zip"
try:
    resposta = requests.get(url_dfp, timeout=300)
    pacote = zipfile.ZipFile(io.BytesIO(resposta.content))
    dre = pd.read_csv(pacote.open("dfp_cia_aberta_DRE_con_2024.csv"),
                      sep=";", encoding="latin-1", dtype=str)
    dre_ultimo = dre[dre["ORDEM_EXERC"] == "ÚLTIMO"].copy()
    dre_ultimo["VL_CONTA"] = pd.to_numeric(dre_ultimo["VL_CONTA"], errors="coerce")
    receita = dre_ultimo[dre_ultimo["CD_CONTA"] == "3.01"][["CD_CVM", "VL_CONTA"]].rename(
        columns={"VL_CONTA": "receita"})
    lucro = dre_ultimo[dre_ultimo["CD_CONTA"] == "3.11"][["CD_CVM", "VL_CONTA"]].rename(
        columns={"VL_CONTA": "lucro_liquido"})
    base = receita.merge(lucro, on="CD_CVM", how="inner").drop_duplicates(subset="CD_CVM")
    url_cad = "https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/cad_cia_aberta.csv"
    cadastro_cvm = pd.read_csv(url_cad, sep=";", encoding="latin-1", dtype=str)
    cadastro_cvm = cadastro_cvm[cadastro_cvm["SIT"] == "ATIVO"][["CD_CVM", "SETOR_ATIV"]]
    base["chave"] = base["CD_CVM"].astype(float).astype(int)
    cadastro_cvm["chave"] = cadastro_cvm["CD_CVM"].astype(float).astype(int)
    base = base.merge(cadastro_cvm[["chave", "SETOR_ATIV"]], on="chave", how="left")
    base = base.rename(columns={"SETOR_ATIV": "setor"}).drop(columns="chave")
    print("Base CVM baixada:", base.shape)
except Exception as e:
    print("Download falhou — use a célula de contingência abaixo.")
    raise

base = base[base["receita"].notna() & (base["receita"] != 0)]
base["margem"] = base["lucro_liquido"] / base["receita"]
base["log_receita"] = np.log(base["receita"])
base.head()

In [ ]:
import os
if "base" not in dir():
    for caminho in ("../../dados/cvm_dre_2024.csv", "cvm_dre_2024.csv"):
        if os.path.exists(caminho):
            base = pd.read_csv(caminho)
            base = base[base["receita"].notna() & (base["receita"] != 0)]
            base["margem"] = base["lucro_liquido"] / base["receita"]
            base["log_receita"] = np.log(base["receita"])
            print("Carregado do arquivo local:", caminho)
            break
print("Base:", base.shape)

In [ ]:
# Escopo da avaliação: três setores empresariais com cadastro suficiente
trab = base[base["setor"].str.contains(
    "Constru|Atacado e Varejo|Energia El", na=False, regex=True)].copy()
trab["setor_curto"] = np.select(
    [trab["setor"].str.contains("Constru", na=False),
     trab["setor"].str.contains("Atacado e Varejo", na=False),
     trab["setor"].str.contains("Energia El", na=False)],
    ["Construção Civil", "Comércio", "Energia Elétrica"],
    default="Outro")
trab = trab[trab["setor_curto"] != "Outro"]
print("Escopo:", trab.shape)
trab["setor_curto"].value_counts()

---
## Tarefa 1 — Estatística descritiva (20 pontos)

**Objetivo:** resumir a **receita** (em milhões de reais) dos três setores e decidir, com
justificativa, entre média e mediana.

**(a)** Complete a função `descritivas` (média, mediana, desvio padrão e coeficiente de
variação) e monte a tabela por setor — na escala de **R$ milhões**.

In [ ]:
def descritivas(serie):
    # === COMPLETE AQUI: média, mediana, desvio padrão (ddof=1) e CV ===
    return pd.Series({
        "n": len(serie),
        "media_milhoes": ...,
        "mediana_milhoes": ...,
        "desvio_milhoes": ...,
        "cv": ...,
    })

tabela = pd.DataFrame()
# === COMPLETE AQUI: aplicar descritivas() à receita (÷1e6) de CADA setor e empilhar ===
...
tabela.round(2)

**(b)** Responda **no texto abaixo** (edite esta célula com suas respostas):

1. Em qual setor a distância entre média e mediana é maior? O que isso revela sobre a
presença de valores extremos?
2. Para **comparar a receita típica** das companhias entre setores, você usaria média ou
mediana? Justifique.
3. O **coeficiente de variação** de qual setor é o maior? O que um CV entre 8 e 16 vezes
a média sugere sobre a confiabilidade da média como resumo desse setor?

*Sua resposta:*

---
## Tarefa 2 — Gráficos (25 pontos)

**(a)** Produza um **boxplot** da margem líquida por setor (com título, rótulos de eixos e
limite de y em [−1,5; 1,5] para a leitura dos extremos):

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
# === COMPLETE AQUI: boxplot da margem por setor_curto, ylim [-1.5; 1.5] ===
...
ax.set_title("Margem líquida por setor (2024)")
plt.show()

**(b)** Produza um **histograma** do log-receita de **todas** as companhias do escopo, com
título e rótulos corretos.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
# === COMPLETE AQUI: histograma de log_receita (bins=40), títulos e rótulos ===
...
plt.show()

**(c)** Responda **no texto abaixo** (edite esta célula):

1. O que o boxplot revela sobre os extremos da margem? Como isso conversa com a Tarefa 1?
2. Por que a transformação logarítmica *parece* normalizar a distribuição da receita?
3. Em um relatório, você apresentaria as margens na escala bruta ou no log? Por quê?

*Sua resposta:*

---
## Tarefa 3 — Inferência estatística (30 pontos)

**Questão de pesquisa:** a **margem líquida média** das companhias de **Energia Elétrica** é
diferente de zero? E a **receita** do **Comércio** difere da da **Construção Civil**?

**(a) Intervalo de confiança** — calcule (passo a passo e via `scipy`) o IC de 95% para a
média da margem de Energia Elétrica:

In [ ]:
energia = trab[trab["setor_curto"] == "Energia Elétrica"]["margem"].dropna()
n = len(energia)
media = energia.mean()
desvio = energia.std(ddof=1)
# === COMPLETE AQUI: t crítico (95%, duas caudas, df = n-1) e margem de erro ===
t_critico = ...
margem_erro = ...
print(f"n = {n} | média = {media:.4f} | desvio = {desvio:.4f}")
print(f"IC (manual) = [{media - margem_erro:.4f} ; {media + margem_erro:.4f}]")

# === COMPLETE AQUI: IC via scipy.stats.t.interval ===
ic = ...
print(f"IC (scipy)  = [{ic[0]:.4f} ; {ic[1]:.4f}]")

**(b)** Ao lado do IC, responda:

1. O intervalo contém o zero? O que isso permite concluir sobre a margem média
(positiva/negativa/não é possível afirmar)?
2. Escreva a **interpretação correta** do que "95% de confiança" significa — sem o erro
comum de dizer que a probabilidade é do parâmetro estar *neste* intervalo.

*Sua resposta:*

**(c) Teste t de Student** — compare a **receita** (na escala log, o motivo é o da Tarefa 2)
das companhias de **Comércio** e **Construção Civil**. Formule H0 e H1, rode o teste com
`scipy.stats.ttest_ind`, decida usando α = 0,05 e **interprete o resultado**.

In [ ]:
comercio = trab[trab["setor_curto"] == "Comércio"]["log_receita"]
construcao = trab[trab["setor_curto"] == "Construção Civil"]["log_receita"]

# === COMPLETE AQUI: teste t de Student entre os dois grupos ===
t_stat, p_valor = ...
print(f"t = {t_stat:.3f} | p = {p_valor:.4f}")

In [ ]:
# COMPLETE o texto: decida com α = 0,05 — a diferença é significativa?
decisao = "..."   # 'rejeita H0' ou 'não rejeita H0'
print("Decisão:", decisao)

**(d)** Responda **no texto abaixo** (edite esta célula):

1. Enuncie H0 e H1 do teste da parte (c).
2. Com base no p-valor, o que você conclui sobre a receita típica dos dois setores?
3. O teste foi feito na escala log. Se fosse feito na **receita bruta**, por que o
resultado poderia ser enganoso? (Dica: lembre-se da Tarefa 1 e dos extremos.)
4. "Não rejeitar H0" equivale a "provar H0"? Explique em uma linha.

*Sua resposta:*

---
## Tarefa 4 — Qui-quadrado de independência (25 pontos)

**Questão de pesquisa:** a **sobrevivência de empresas** de 1 ano (CEMPRE/IBGE, tabela 9949)
é associada ao **porte** (faixa de pessoal ocupado)? Execute a preparação dos dados:

In [ ]:
import sidrapy

def limpa_sidra(df):
    df = df.copy()
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)
    df["Valor"] = pd.to_numeric(df["Valor"], errors="coerce")
    return df

try:
    bruto = sidrapy.get_table(
        table_code="9949", territorial_level="1", ibge_territorial_code="all",
        variable="all", classifications={"12762": "all", "370": "all"}, period="all")
    demo = limpa_sidra(bruto)
except Exception:
    for caminho in ("../../dados/demografia_sobrevivencia_empresas.csv",
                    "demografia_sobrevivencia_empresas.csv"):
        if os.path.exists(caminho):
            demo = pd.read_csv(caminho, dtype=str)
            demo = limpa_sidra(demo)
            break

demo["D3N"] = demo["D3N"].astype(str).str.replace(",", ".")
demo["D4N"] = demo["D4N"].astype(str).str.replace(",", ".")
demo = demo.sort_values("Ano")

**(a)** Monte a tabela de contingência **sobreviventes × não sobreviventes por faixa de
pessoal** para o ano mais recente, com as colunas `D5N` (faixa), `nascimentos`,
`taxa_sobrev` (a taxa já vem em %), `sobreviveram = nascimentos × taxa / 100` e
`nao_sobreviveram`.

In [ ]:
# === COMPLETE AQUI: filtrar variável de nascimentos (D2N) do ano mais recente e calcular
#     sobreviveram / nao_sobreviveram por faixa D5N ===
tabela4 = ...
tabela4

**(b)** Rode o **teste qui-quadrado de independência** (`scipy.stats.chi2_contingency`)
sobre a tabela de contingência e responda no texto:

1. Qual a estatística (χ²), quantos graus de liberdade e o p-valor?
2. Qual a conclusão, com α = 0,05, sobre a associação entre porte e sobrevivência?
3. Que **limitação** esse teste carrega aqui? (O que os dados permitem afirmar — e o que
não permitem?)

In [ ]:
# === COMPLETE AQUI: matriz de contingência (sobreviveram, nao_sobreviveram) e chi2 ===
contingencia = ...
chi2, p, dof, _ = stats.chi2_contingency(contingencia)
print(f"χ² = {chi2:.1f} | dof = {dof} | p = {p:.2e}")

---
## Antes de entregar

1. **Ambiente de execução → Reiniciar e executar tudo** — confirme que nada quebra;
2. Confira as **três tarefas** (células completadas, gabaritos de resposta escritos);
3. Salve e **compartilhe o link** com o professor com permissão de edição;
4. Revise a interpretação de cada resultado — ela vale metade dos pontos.